In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import torch

DATA_ROOT = "final_data"
feat_path = f"{DATA_ROOT}/MGTAB/features.csv"
label_path = f"{DATA_ROOT}/MGTAB/labels_bot.csv"
edge_path = f"{DATA_ROOT}/MGTAB/edge_index.csv"

# Загружаем признаки и метки
df_feat = pd.read_csv(feat_path, header=None, low_memory=False)
df_labels = pd.read_csv(label_path, header=None)
y = df_labels[0].values
X = df_feat.values

N = len(X)
d = X.shape[1]
pos = (y == 1).sum()
neg = (y == 0).sum()
IR = neg / pos
log10_IR = np.log10(IR)

print(f"N = {N}")
print(f"d = {d}")
print(f"pos = {pos}, neg = {neg}, IR = {IR:.4f}, log10(IR) = {log10_IR:.4f}")

# Загружаем граф
df_edges = pd.read_csv(edge_path, header=None, low_memory=False)
sources = df_edges.iloc[1].values.astype(np.int64)
targets = df_edges.iloc[2].values.astype(np.int64)

# Строим граф в networkx
G = nx.Graph()
G.add_edges_from(zip(sources, targets))

# Средняя степень
degrees = [deg for _, deg in G.degree()]
avg_deg = np.mean(degrees)

# Коэффициент кластеризации (только для узлов со степенью >=2, но среднее по всем)
cc = nx.average_clustering(G)

print(f"Средняя степень: {avg_deg:.4f}")
print(f"Средний коэффициент кластеризации: {cc:.4f}")

# Также можно вывести количество узлов по графу (должно быть 10200 или больше)
num_nodes_graph = G.number_of_nodes()
print(f"Узлов в графе: {num_nodes_graph}")

N = 10200
d = 788
pos = 2748, neg = 7452, IR = 2.7118, log10(IR) = 0.4333
Средняя степень: 179.0598
Средний коэффициент кластеризации: 0.2215
Узлов в графе: 10147


In [2]:
import pandas as pd
import numpy as np
import os
import networkx as nx
import torch

DATA_ROOT = "final_data"

# ==================== ФУНКЦИИ ЗАГРУЗКИ (КОПИЯ ИЗ ЭКСПЕРИМЕНТОВ) ====================
def load_creditcard():
    df = pd.read_csv(os.path.join(DATA_ROOT, "creditcard.csv"))
    return df, 'Class'

def load_ieee():
    df = pd.read_csv(os.path.join(DATA_ROOT, "ieee_train_transaction.csv"))
    return df, 'isFraud'

def load_paysim():
    df = pd.read_csv(os.path.join(DATA_ROOT, "paysim.csv"))
    return df, 'isFraud'

def load_aizbank():
    df = pd.read_csv(os.path.join(DATA_ROOT, "aizbank.csv"))
    return df, 'label'

def load_elliptic():
    feat_path = os.path.join(DATA_ROOT, "Elliptic", "elliptic_txs_features.csv")
    class_path = os.path.join(DATA_ROOT, "Elliptic", "elliptic_txs_classes.csv")
    df_feat = pd.read_csv(feat_path, header=None, low_memory=False)
    df_feat.columns = ['txId', 'time_step'] + [f'f{i}' for i in range(df_feat.shape[1]-2)]
    df_class = pd.read_csv(class_path)
    df_class = df_class[df_class['class'].isin(['1','2'])].copy()
    df_class['label'] = df_class['class'].map({'1':0, '2':1})
    df = df_feat.merge(df_class[['txId', 'label']], on='txId', how='inner')
    return df, 'label'

def load_amazon():
    df = pd.read_csv(os.path.join(DATA_ROOT, "Amazon", "Amazon_full.csv"), header=None, skiprows=1, low_memory=False)
    df.columns = [f'feat_{i}' for i in range(df.shape[1]-1)] + ['label']
    df['label'] = df['label'].astype(float).astype(int)
    return df, 'label'

def load_yelp():
    df = pd.read_csv(os.path.join(DATA_ROOT, "YelpChi", "YelpChi_full.csv"))
    return df, 'label'

def load_click_fraud():
    df = pd.read_csv(os.path.join(DATA_ROOT, "click_fraud_sample.csv"))
    return df, 'is_attributed'

def load_social_honeypot():
    df = pd.read_csv(os.path.join(DATA_ROOT, "social_honeypot.csv"))
    return df, 'label'

def load_cresci():
    df = pd.read_csv(os.path.join(DATA_ROOT, "cresci_merged.csv"))
    return df, 'label'

def load_midterm():
    df = pd.read_csv(os.path.join(DATA_ROOT, "midterm_merged.csv"))
    return df, 'label'

def load_mgtab():
    feat_path = os.path.join(DATA_ROOT, "MGTAB", "features.csv")
    label_path = os.path.join(DATA_ROOT, "MGTAB", "labels_bot.csv")
    edge_path = os.path.join(DATA_ROOT, "MGTAB", "edge_index.csv")
    df_feat = pd.read_csv(feat_path, header=None, low_memory=False)
    X = df_feat.values.astype(np.float32)
    df_labels = pd.read_csv(label_path, header=None)
    y = df_labels[0].values.astype(int)
    df_edges = pd.read_csv(edge_path, header=None, low_memory=False)
    sources = df_edges.iloc[1].values.astype(np.int64)
    targets = df_edges.iloc[2].values.astype(np.int64)
    edge_index = torch.tensor([sources, targets], dtype=torch.long)
    num_nodes_edge = edge_index.max().item() + 1
    if X.shape[0] != num_nodes_edge:
        if X.shape[0] < num_nodes_edge:
            pad = np.zeros((num_nodes_edge - X.shape[0], X.shape[1]), dtype=np.float32)
            X = np.vstack([X, pad])
        else:
            X = X[:num_nodes_edge]
    if len(y) != num_nodes_edge:
        if len(y) < num_nodes_edge:
            pad_y = np.zeros(num_nodes_edge - len(y), dtype=np.int64)
            y = np.concatenate([y, pad_y])
        else:
            y = y[:num_nodes_edge]
    df = pd.DataFrame(X)
    df['label'] = y
    return df, 'label'

def load_pronbots():
    # Не используется
    return pd.DataFrame(), None

# ==================== СЛОВАРЬ ДАТАСЕТОВ ====================
datasets_config = {
    "Credit Card Fraud": load_creditcard,
    "IEEE-CIS": load_ieee,
    "PaySim": load_paysim,
    "AIZBank": load_aizbank,
    "Elliptic (Bitcoin)": load_elliptic,
    "Amazon Reviews": load_amazon,
    "Yelp Reviews": load_yelp,
    "Click Fraud": load_click_fraud,
    "Social Honeypot": load_social_honeypot,
    "MGTAB": load_mgtab,
    "Cresci-2017": load_cresci,
    "Midterm-2018": load_midterm,
}

# ==================== ВЫЧИСЛЕНИЕ МЕТА-ПРИЗНАКОВ ====================
def compute_meta_features(name, loader_func):
    print(f"\nОбработка: {name}")
    try:
        df, target_col = loader_func()
    except Exception as e:
        print(f"  Ошибка загрузки: {e}")
        return None
    if df is None or len(df) == 0:
        return None
    N = len(df)
    logN = np.log10(N)
    if target_col and target_col in df.columns:
        y = df[target_col]
        pos = (y == 1).sum()
        neg = (y == 0).sum()
        if pos > 0 and neg > 0:
            ir = neg / pos
            logIR = np.log10(ir)
        else:
            ir = np.nan
            logIR = np.nan
        d = df.shape[1] - 1
    else:
        d = df.shape[1]
        ir = np.nan
        logIR = np.nan
    # Определяем наличие графа (по имени датасета, если есть графовые файлы)
    G_flag = 0
    avg_deg = None
    cc = None
    if name in ["Elliptic (Bitcoin)", "Amazon Reviews", "Yelp Reviews", "MGTAB"]:
        G_flag = 1
        # Пытаемся вычислить графовые метрики
        try:
            if name == "Elliptic (Bitcoin)":
                edge_path = os.path.join(DATA_ROOT, "Elliptic", "elliptic_txs_edgelist.csv")
                edges = pd.read_csv(edge_path, header=None).values
                # Переиндексация не нужна, считаем по всем узлам? Но может быть много узлов без признаков.
                # Для простоты строим граф по всем рёбрам и вычисляем степень и кластеризацию
                G = nx.Graph()
                # Узлы – все встречающиеся идентификаторы
                # Но для метрики лучше использовать подмножество, но для приближения сойдёт
                unique_nodes = set(edges.flatten())
                G.add_nodes_from(unique_nodes)
                G.add_edges_from(edges)
                if G.number_of_nodes() > 0:
                    avg_deg = 2 * G.number_of_edges() / G.number_of_nodes()
                    cc = nx.average_clustering(G)
                else:
                    avg_deg = 0
                    cc = 0
            elif name == "Amazon Reviews":
                edge_path = os.path.join(DATA_ROOT, "Amazon", "Amazon_net_uvu_edges.csv")
                edges = pd.read_csv(edge_path)[['row', 'col']].values
                G = nx.Graph()
                G.add_edges_from(edges)
                avg_deg = 2 * G.number_of_edges() / G.number_of_nodes()
                cc = nx.average_clustering(G)
            elif name == "Yelp Reviews":
                edge_path = os.path.join(DATA_ROOT, "YelpChi", "YelpChi_net_rur_edges.csv")
                edges = pd.read_csv(edge_path)[['row', 'col']].values
                G = nx.Graph()
                G.add_edges_from(edges)
                avg_deg = 2 * G.number_of_edges() / G.number_of_nodes()
                cc = nx.average_clustering(G)
            elif name == "MGTAB":
                edge_path = os.path.join(DATA_ROOT, "MGTAB", "edge_index.csv")
                df_edges = pd.read_csv(edge_path, header=None, low_memory=False)
                sources = df_edges.iloc[1].values.astype(np.int64)
                targets = df_edges.iloc[2].values.astype(np.int64)
                G = nx.Graph()
                G.add_edges_from(zip(sources, targets))
                avg_deg = 2 * G.number_of_edges() / G.number_of_nodes()
                cc = nx.average_clustering(G)
        except Exception as e:
            print(f"  Ошибка вычисления графовых метрик: {e}")
            avg_deg = None
            cc = None
    result = {
        "Dataset": name,
        "log10(N)": round(logN, 2),
        "log10(IR)": round(logIR, 2) if not np.isnan(logIR) else "—",
        "d": d,
        "G": G_flag,
        "¯deg": round(avg_deg, 2) if avg_deg is not None else "—",
        "CC": round(cc, 4) if cc is not None else "—"
    }
    print(f"  N={N}, d={d}, IR={ir}, G={G_flag}, deg={avg_deg}, CC={cc}")
    return result

# ==================== ЗАПУСК ====================
results = []
for name, loader in datasets_config.items():
    meta = compute_meta_features(name, loader)
    if meta:
        results.append(meta)

df_meta = pd.DataFrame(results)
print("\n" + "="*80)
print("МЕТА-ПРИЗНАКИ")
print("="*80)
print(df_meta.to_string(index=False))
df_meta.to_csv("meta_features_complete.csv", index=False)
print("\nСохранено в meta_features_complete.csv")


Обработка: Credit Card Fraud
  N=284807, d=30, IR=577.8760162601626, G=0, deg=None, CC=None

Обработка: IEEE-CIS
  N=590540, d=393, IR=27.579586700866283, G=0, deg=None, CC=None

Обработка: PaySim
  N=6362620, d=10, IR=773.7010836478753, G=0, deg=None, CC=None

Обработка: AIZBank
  N=45211, d=16, IR=7.548118737001324, G=0, deg=None, CC=None

Обработка: Elliptic (Bitcoin)
  N=46564, d=167, IR=0.10816535376853328, G=1, deg=2.3001899190758253, CC=0.013762055649177894

Обработка: Amazon Reviews
  N=11944, d=25, IR=13.548112058465286, G=1, deg=174.78496164545226, CC=0.6606880837282073

Обработка: Yelp Reviews
  N=45954, d=32, IR=5.882432230043433, G=1, deg=4.138726868364735, CC=0.6579245520540472

Обработка: Click Fraud
  N=100000, d=7, IR=439.52863436123346, G=0, deg=None, CC=None

Обработка: Social Honeypot
  N=41499, d=6, IR=0.8673896413625524, G=0, deg=None, CC=None

Обработка: MGTAB


/var/folders/md/78mwsg9n23525_j8040kg6fw0000gn/T/ipykernel_3720/2225454223.py:74: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  edge_index = torch.tensor([sources, targets], dtype=torch.long)


  N=10199, d=788, IR=2.7114264919941777, G=1, deg=179.05982063664138, CC=0.22150735337337427

Обработка: Cresci-2017
  N=6105, d=21, IR=1.3204104903078677, G=0, deg=None, CC=None

Обработка: Midterm-2018
  N=50538, d=19, IR=0.19064222777175704, G=0, deg=None, CC=None

МЕТА-ПРИЗНАКИ
           Dataset  log10(N)  log10(IR)   d  G    ¯deg      CC
 Credit Card Fraud      5.45       2.76  30  0       —       —
          IEEE-CIS      5.77       1.44 393  0       —       —
            PaySim      6.80       2.89  10  0       —       —
           AIZBank      4.66       0.88  16  0       —       —
Elliptic (Bitcoin)      4.67      -0.97 167  1     2.3  0.0138
    Amazon Reviews      4.08       1.13  25  1  174.78  0.6607
      Yelp Reviews      4.66       0.77  32  1    4.14  0.6579
       Click Fraud      5.00       2.64   7  0       —       —
   Social Honeypot      4.62      -0.06   6  0       —       —
             MGTAB      4.01       0.43 788  1  179.06  0.2215
       Cresci-2017      

/var/folders/md/78mwsg9n23525_j8040kg6fw0000gn/T/ipykernel_3720/2225454223.py:60: DtypeWarning: Columns (8,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(DATA_ROOT, "midterm_merged.csv"))


In [ ]:
import pandas as pd

# Загружаем файл без заголовков
df = pd.read_csv("data/midterm-2018/midterm-2018.tsv", sep='\t', header=None)

# Первый столбец (индекс 0) — id, второй (индекс 1) — метка
labels = df[1].value_counts()

print("Распределение меток во втором столбце:")
print(labels)

# Если метки называются 'human' и 'bot'
human_count = labels.get('human', 0)
bot_count = labels.get('bot', 0)

print(f"\nHuman: {human_count}")
print(f"Bot: {bot_count}")
print(f"Всего: {human_count + bot_count}")

In [1]:
import pandas as pd
import numpy as np
import os
import shutil

# ==================== КОНФИГУРАЦИЯ ====================
SOURCE_ROOT = "data"       # исходная папка (где лежат原始数据)
TARGET_ROOT = "final_data" # целевая папка для обработанных данных

# ==================== ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ====================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def copy_file(src, dst):
    ensure_dir(os.path.dirname(dst))
    shutil.copy2(src, dst)
    print(f"  Копирован: {src} -> {dst}")

def copy_folder(src, dst):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"  Скопирована папка: {src} -> {dst}")

# ==================== ОБРАБОТКА ДАТАСЕТОВ ====================
def process_creditcard():
    src = os.path.join(SOURCE_ROOT, "creditcard.csv")
    dst = os.path.join(TARGET_ROOT, "creditcard.csv")
    copy_file(src, dst)

def process_ieee():
    src = os.path.join(SOURCE_ROOT, "IEEE-CIS", "train_transaction.csv")
    dst = os.path.join(TARGET_ROOT, "ieee_train_transaction.csv")
    copy_file(src, dst)

def process_paysim():
    src = os.path.join(SOURCE_ROOT, "PaySim.csv")
    dst = os.path.join(TARGET_ROOT, "paysim.csv")
    copy_file(src, dst)

def process_aizbank():
    src = os.path.join(SOURCE_ROOT, "AIZBank-full.csv")
    dst = os.path.join(TARGET_ROOT, "aizbank.csv")
    df = pd.read_csv(src)
    # Добавляем колонку label на основе 'class'
    df['label'] = df['class'].map({'no': 0, 'yes': 1})
    # Удаляем исходную колонку 'class', чтобы не было путаницы (можно и оставить)
    df = df.drop(columns=['class'])
    ensure_dir(os.path.dirname(dst))
    df.to_csv(dst, index=False)
    print(f"  Обработан AIZBank: {dst} (добавлена колонка label)")

def process_elliptic():
    src_folder = os.path.join(SOURCE_ROOT, "Elliptic")
    dst_folder = os.path.join(TARGET_ROOT, "Elliptic")
    copy_folder(src_folder, dst_folder)

def process_amazon():
    src_folder = os.path.join(SOURCE_ROOT, "Amazon")
    dst_folder = os.path.join(TARGET_ROOT, "Amazon")
    copy_folder(src_folder, dst_folder)

def process_yelp():
    src_folder = os.path.join(SOURCE_ROOT, "YelpChi")
    dst_folder = os.path.join(TARGET_ROOT, "YelpChi")
    copy_folder(src_folder, dst_folder)

def process_click_fraud():
    src = os.path.join(SOURCE_ROOT, "TalkingData", "train_sample.csv")
    dst = os.path.join(TARGET_ROOT, "click_fraud_sample.csv")
    copy_file(src, dst)

def process_social_honeypot():
    cp_src = os.path.join(SOURCE_ROOT, "SHD", "content_polluters.csv")
    legit_src = os.path.join(SOURCE_ROOT, "SHD", "legitimate_users.csv")
    dst = os.path.join(TARGET_ROOT, "social_honeypot.csv")
    cp = pd.read_csv(cp_src)
    legit = pd.read_csv(legit_src)
    cp['label'] = 1
    legit['label'] = 0
    df = pd.concat([cp, legit], ignore_index=True)
    # Оставляем только числовые колонки (включая label)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df = df[numeric_cols]
    ensure_dir(os.path.dirname(dst))
    df.to_csv(dst, index=False)
    print(f"  Обработан Social Honeypot: {dst}")

def process_mgtab():
    src_folder = os.path.join(SOURCE_ROOT, "MGTAB")
    dst_folder = os.path.join(TARGET_ROOT, "MGTAB")
    # Копируем папку целиком (там должны быть edge_index.csv, labels_stance.csv, возможно features.csv)
    copy_folder(src_folder, dst_folder)
    # Если файл признаков отсутствует, можно будет добавить позже вручную.

def process_cresci():
    # Объединяем genuine и traditional_spambots
    target_file = os.path.join(TARGET_ROOT, "cresci_merged.csv")
    all_dfs = []
    # genuine
    genuine_path = os.path.join(SOURCE_ROOT, "Cresci-2017", "genuine_accounts.csv", "users.csv")
    if os.path.exists(genuine_path):
        try:
            genuine = pd.read_csv(genuine_path, encoding='utf-8', low_memory=False)
        except:
            genuine = pd.read_csv(genuine_path, encoding='latin1', low_memory=False)
        genuine['label'] = 0
        all_dfs.append(genuine)
        print(f"  Загружены genuine: {len(genuine)} записей")
    # traditional spambots
    spambot_folders = [
        "traditional_spambots_1.csv",
        "traditional_spambots_2.csv",
        "traditional_spambots_3.csv",
        "traditional_spambots_4.csv"
    ]
    for folder in spambot_folders:
        spambot_path = os.path.join(SOURCE_ROOT, "Cresci-2017", folder, "users.csv")
        if os.path.exists(spambot_path):
            try:
                spambot = pd.read_csv(spambot_path, encoding='utf-8', low_memory=False)
            except:
                spambot = pd.read_csv(spambot_path, encoding='latin1', low_memory=False)
            spambot['label'] = 1
            all_dfs.append(spambot)
            print(f"  Загружены {folder}: {len(spambot)} записей")
    if all_dfs:
        df = pd.concat(all_dfs, ignore_index=True)
        # Оставляем только числовые колонки + label
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if 'label' not in numeric_cols:
            numeric_cols.append('label')
        df = df[numeric_cols]
        ensure_dir(os.path.dirname(target_file))
        df.to_csv(target_file, index=False)
        print(f"  Сохранён объединённый Cresci-2017: {df.shape} -> {target_file}")
    else:
        print("  Ошибка: не найдены файлы Cresci-2017")

def process_midterm():
    src_csv = os.path.join(SOURCE_ROOT, "midterm-2018", "midterm-2018_processed_user_objects.csv")
    src_tsv = os.path.join(SOURCE_ROOT, "midterm-2018", "midterm-2018.tsv")
    dst = os.path.join(TARGET_ROOT, "midterm_merged.csv")
    if not os.path.exists(src_csv):
        print(f"  Ошибка: не найден {src_csv}")
        return
    if not os.path.exists(src_tsv):
        print(f"  Ошибка: не найден {src_tsv}")
        return
    # Загрузка признаков
    df_features = pd.read_csv(src_csv, engine='python', encoding='utf-8', on_bad_lines='skip')
    print(f"  Загружены признаки: {len(df_features)} строк")
    # Загрузка меток
    df_labels = pd.read_csv(src_tsv, sep='\t', header=None, names=['user_id', 'label_raw'])
    df_labels['label'] = df_labels['label_raw'].map({'bot': 1, 'human': 0})
    df_labels = df_labels.drop(columns=['label_raw'])
    print(f"  Загружены метки: {len(df_labels)} строк")
    # Объединение
    df_features['user_id'] = df_features['user_id'].astype(str)
    df_labels['user_id'] = df_labels['user_id'].astype(str)
    df = df_features.merge(df_labels, on='user_id', how='inner')
    print(f"  После объединения: {len(df)} строк")
    ensure_dir(os.path.dirname(dst))
    df.to_csv(dst, index=False)
    print(f"  Сохранён объединённый Midterm-2018: {dst}")

def process_pronbots():
    src = os.path.join(SOURCE_ROOT, "pronbots-2019", "pronbots-2019_tweets.csv")
    dst = os.path.join(TARGET_ROOT, "pronbots_tweets.csv")
    copy_file(src, dst)

# ==================== СЛОВАРЬ ПРОЦЕССОРОВ ====================
processors = [
    ("Credit Card Fraud", process_creditcard),
    ("IEEE-CIS", process_ieee),
    ("PaySim", process_paysim),
    ("AIZBank", process_aizbank),
    ("Elliptic (Bitcoin)", process_elliptic),
    ("Amazon Reviews", process_amazon),
    ("Yelp Reviews", process_yelp),
    ("Click Fraud", process_click_fraud),
    ("Social Honeypot", process_social_honeypot),
    ("MGTAB", process_mgtab),
    ("Cresci-2017", process_cresci),
    ("Midterm-2018", process_midterm),
    ("Pronbots-2019", process_pronbots),
]

# ==================== ЗАПУСК ====================
if __name__ == "__main__":
    print("="*80)
    print("ПОДГОТОВКА ДАННЫХ ДЛЯ FINAL_DATA")
    print("="*80)
    ensure_dir(TARGET_ROOT)
    for name, proc in processors:
        print(f"\nОбработка: {name}")
        proc()
    print("\n" + "="*80)
    print(f"ГОТОВО! Все датасеты сохранены в папку '{TARGET_ROOT}'")
    print("="*80)

ПОДГОТОВКА ДАННЫХ ДЛЯ FINAL_DATA

Обработка: Credit Card Fraud
  Копирован: data/creditcard.csv -> final_data/creditcard.csv

Обработка: IEEE-CIS
  Копирован: data/IEEE-CIS/train_transaction.csv -> final_data/ieee_train_transaction.csv

Обработка: PaySim
  Копирован: data/PaySim.csv -> final_data/paysim.csv

Обработка: AIZBank
  Обработан AIZBank: final_data/aizbank.csv (добавлена колонка label)

Обработка: Elliptic (Bitcoin)
  Скопирована папка: data/Elliptic -> final_data/Elliptic

Обработка: Amazon Reviews
  Скопирована папка: data/Amazon -> final_data/Amazon

Обработка: Yelp Reviews
  Скопирована папка: data/YelpChi -> final_data/YelpChi

Обработка: Click Fraud
  Копирован: data/TalkingData/train_sample.csv -> final_data/click_fraud_sample.csv

Обработка: Social Honeypot
  Обработан Social Honeypot: final_data/social_honeypot.csv

Обработка: MGTAB
  Скопирована папка: data/MGTAB -> final_data/MGTAB

Обработка: Cresci-2017
  Загружены genuine: 3474 записей
  Загружены traditional_sp

In [2]:
import platform
import psutil
import multiprocessing

print("="*50)
print("ХАРАКТЕРИСТИКИ ВЫЧИСЛИТЕЛЬНОЙ СРЕДЫ")
print("="*50)
print(f"ОС: {platform.system()} {platform.release()}")
print(f"Процессор: {platform.processor()}")
print(f"Ядра CPU: {multiprocessing.cpu_count()}")
print(f"ОЗУ: {psutil.virtual_memory().total / (1024**3):.1f} ГБ")
print(f"Среда выполнения: PyCharm на локальной машине (CPU)")

ХАРАКТЕРИСТИКИ ВЫЧИСЛИТЕЛЬНОЙ СРЕДЫ
ОС: Darwin 24.6.0
Процессор: arm
Ядра CPU: 10
ОЗУ: 16.0 ГБ
Среда выполнения: PyCharm на локальной машине (CPU)


In [4]:
import pandas as pd
import numpy as np

# ==================== ЗАГРУЗКА РЕЗУЛЬТАТОВ ИЗ ФАЙЛОВ ====================
try:
    lr_results = pd.read_csv("lr_results.csv")
    rf_results = pd.read_csv("rf_results.csv")
    xgb_results = pd.read_csv("xgb_results.csv")
    mlp_results = pd.read_csv("mlp_results.csv")
    vae_results = pd.read_csv("vae_results.csv")
    graphsage_results = pd.read_csv("graphsage_results.csv")
    gcn_results = pd.read_csv("gcn_results.csv")
    gat_results = pd.read_csv("gat_results.csv")
    print("Все файлы успешно загружены")
except FileNotFoundError as e:
    print(f"Ошибка: {e}")
    exit(1)

# Функция форматирования
def fmt(m, s):
    if pd.isna(m) or pd.isna(s):
        return "-"
    return f"{m:.4f} $\\pm$ {s:.4f}" if s > 0 else f"{m:.4f}"

def fmt_time(t):
    if pd.isna(t):
        return "-"
    return f"{t:.2f}"

# Списки датасетов
tabular_datasets = ["Credit Card Fraud", "IEEE-CIS", "PaySim", "AIZBank",
                    "Elliptic (Bitcoin)", "Amazon Reviews", "Yelp Reviews",
                    "Click Fraud", "Social Honeypot", "Cresci-2017", "Midterm-2018"]

graph_datasets = ["Elliptic (Bitcoin)", "Amazon Reviews", "Yelp Reviews", "MGTAB"]

# ==================== 6.1 AUPRC ====================
def generate_auprc_table():
    methods = ["LR", "RF", "XGBoost", "MLP", "VAE"]
    latex = "\\begin{table}[H]\n\\centering\n\\caption{Сравнение AUPRC для табличных методов}\n"
    latex += "\\label{tab:auprc_tabular}\n\\footnotesize\n\\setlength{\\tabcolsep}{4pt}\n"
    latex += "\\begin{tabular}{l" + "c"*len(methods) + "}\n\\toprule\n"
    latex += "\\textbf{Датасет} & " + " & ".join(methods) + " \\\\\n\\midrule\n"
    for ds in tabular_datasets:
        row = ds
        lr_r = lr_results[lr_results["Dataset"] == ds]
        rf_r = rf_results[rf_results["Dataset"] == ds]
        xgb_r = xgb_results[xgb_results["Dataset"] == ds]
        mlp_r = mlp_results[mlp_results["Dataset"] == ds]
        vae_r = vae_results[vae_results["Dataset"] == ds]

        row += f" & {fmt(lr_r['AUPRC_mean'].values[0], lr_r['AUPRC_std'].values[0]) if not lr_r.empty else '-'}"
        row += f" & {fmt(rf_r['AUPRC_mean'].values[0], rf_r['AUPRC_std'].values[0]) if not rf_r.empty else '-'}"
        row += f" & {fmt(xgb_r['AUPRC_mean'].values[0], xgb_r['AUPRC_std'].values[0]) if not xgb_r.empty else '-'}"
        row += f" & {fmt(mlp_r['AUPRC_mean'].values[0], mlp_r['AUPRC_std'].values[0]) if not mlp_r.empty else '-'}"
        row += f" & {fmt(vae_r['AUPRC_mean'].values[0], vae_r['AUPRC_std'].values[0]) if not vae_r.empty else '-'}"
        row += " \\\\\n"
        latex += row
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    return latex

# ==================== 6.2 F1 ====================
def generate_f1_table():
    methods = ["LR", "RF", "XGBoost", "MLP"]
    latex = "\\begin{table}[H]\n\\centering\n\\caption{Сравнение F1-меры для табличных методов}\n"
    latex += "\\label{tab:f1_tabular}\n\\footnotesize\n\\setlength{\\tabcolsep}{4pt}\n"
    latex += "\\begin{tabular}{l" + "c"*len(methods) + "}\n\\toprule\n"
    latex += "\\textbf{Датасет} & " + " & ".join(methods) + " \\\\\n\\midrule\n"
    for ds in tabular_datasets:
        row = ds
        lr_r = lr_results[lr_results["Dataset"] == ds]
        rf_r = rf_results[rf_results["Dataset"] == ds]
        xgb_r = xgb_results[xgb_results["Dataset"] == ds]
        mlp_r = mlp_results[mlp_results["Dataset"] == ds]

        row += f" & {fmt(lr_r['F1_mean'].values[0], lr_r['F1_std'].values[0]) if not lr_r.empty else '-'}"
        row += f" & {fmt(rf_r['F1_mean'].values[0], rf_r['F1_std'].values[0]) if not rf_r.empty else '-'}"
        row += f" & {fmt(xgb_r['F1_mean'].values[0], xgb_r['F1_std'].values[0]) if not xgb_r.empty else '-'}"
        row += f" & {fmt(mlp_r['F1_mean'].values[0], mlp_r['F1_std'].values[0]) if not mlp_r.empty else '-'}"
        row += " \\\\\n"
        latex += row
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    return latex

# ==================== 6.3 Precision@K ====================
def generate_preck_table():
    methods = ["LR", "RF", "XGBoost", "MLP", "VAE", "GraphSAGE", "GCN", "GAT"]
    latex = "\\begin{table}[H]\n\\centering\n\\caption{Сравнение Precision@K для всех методов}\n"
    latex += "\\label{tab:preck_all}\n\\footnotesize\n\\setlength{\\tabcolsep}{3pt}\n"
    latex += "\\begin{tabular}{l" + "c"*len(methods) + "}\n\\toprule\n"
    latex += "\\textbf{Датасет} & " + " & ".join(methods) + " \\\\\n\\midrule\n"
    for ds in tabular_datasets:
        lr_r = lr_results[lr_results["Dataset"] == ds]
        rf_r = rf_results[rf_results["Dataset"] == ds]
        xgb_r = xgb_results[xgb_results["Dataset"] == ds]
        mlp_r = mlp_results[mlp_results["Dataset"] == ds]
        vae_r = vae_results[vae_results["Dataset"] == ds]
        row = ds
        row += f" & {fmt(lr_r['Prec@K_mean'].values[0], lr_r['Prec@K_std'].values[0]) if not lr_r.empty else '-'}"
        row += f" & {fmt(rf_r['Prec@K_mean'].values[0], rf_r['Prec@K_std'].values[0]) if not rf_r.empty else '-'}"
        row += f" & {fmt(xgb_r['Prec@K_mean'].values[0], xgb_r['Prec@K_std'].values[0]) if not xgb_r.empty else '-'}"
        row += f" & {fmt(mlp_r['Prec@K_mean'].values[0], mlp_r['Prec@K_std'].values[0]) if not mlp_r.empty else '-'}"
        row += f" & {fmt(vae_r['Prec@K_mean'].values[0], vae_r['Prec@K_std'].values[0]) if not vae_r.empty else '-'}"
        gs_r = graphsage_results[graphsage_results["Dataset"] == ds]
        row += f" & {fmt(gs_r['Prec@K_mean'].values[0], gs_r['Prec@K_std'].values[0]) if not gs_r.empty else '-'}"
        gcn_r = gcn_results[gcn_results["Dataset"] == ds]
        row += f" & {fmt(gcn_r['Prec@K_mean'].values[0], gcn_r['Prec@K_std'].values[0]) if not gcn_r.empty else '-'}"
        gat_r = gat_results[gat_results["Dataset"] == ds]
        row += f" & {fmt(gat_r['Prec@K_mean'].values[0], gat_r['Prec@K_std'].values[0]) if not gat_r.empty else '-'}"
        row += " \\\\\n"
        latex += row
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    return latex

# ==================== 6.4 ДОМЕНЫ ====================
def generate_domain_tables():
    financial = ["Credit Card Fraud", "IEEE-CIS", "PaySim", "AIZBank", "Elliptic (Bitcoin)"]
    ecommerce = ["Amazon Reviews", "Yelp Reviews", "Click Fraud"]
    social = ["Social Honeypot", "Cresci-2017", "Midterm-2018", "MGTAB"]
    methods = ["LR", "RF", "XGBoost", "MLP", "VAE"]
    graph_methods = ["GraphSAGE", "GCN", "GAT"]

    latex = "\\subsubsection{Финансовые транзакции}\n\n"
    latex += "\\begin{table}[H]\n\\centering\n\\caption{AUPRC на финансовых датасетах}\n"
    latex += "\\label{tab:domain_financial}\n\\footnotesize\n\\setlength{\\tabcolsep}{4pt}\n"
    latex += "\\begin{tabular}{l" + "c"*len(methods) + "}\n\\toprule\n"
    latex += "\\textbf{Датасет} & " + " & ".join(methods) + " \\\\\n\\midrule\n"
    for ds in financial:
        lr_r = lr_results[lr_results["Dataset"] == ds]
        rf_r = rf_results[rf_results["Dataset"] == ds]
        xgb_r = xgb_results[xgb_results["Dataset"] == ds]
        mlp_r = mlp_results[mlp_results["Dataset"] == ds]
        vae_r = vae_results[vae_results["Dataset"] == ds]
        row = ds
        row += f" & {fmt(lr_r['AUPRC_mean'].values[0], lr_r['AUPRC_std'].values[0]) if not lr_r.empty else '-'}"
        row += f" & {fmt(rf_r['AUPRC_mean'].values[0], rf_r['AUPRC_std'].values[0]) if not rf_r.empty else '-'}"
        row += f" & {fmt(xgb_r['AUPRC_mean'].values[0], xgb_r['AUPRC_std'].values[0]) if not xgb_r.empty else '-'}"
        row += f" & {fmt(mlp_r['AUPRC_mean'].values[0], mlp_r['AUPRC_std'].values[0]) if not mlp_r.empty else '-'}"
        row += f" & {fmt(vae_r['AUPRC_mean'].values[0], vae_r['AUPRC_std'].values[0]) if not vae_r.empty else '-'}"
        row += " \\\\\n"
        latex += row
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"

    latex += "\n\\subsubsection{Электронная коммерция}\n\n"
    latex += "\\begin{table}[H]\n\\centering\n\\caption{AUPRC на датасетах электронной коммерции}\n"
    latex += "\\label{tab:domain_ecommerce}\n\\footnotesize\n\\setlength{\\tabcolsep}{4pt}\n"
    latex += "\\begin{tabular}{l" + "c"*len(methods) + "}\n\\toprule\n"
    latex += "\\textbf{Датасет} & " + " & ".join(methods) + " \\\\\n\\midrule\n"
    for ds in ecommerce:
        lr_r = lr_results[lr_results["Dataset"] == ds]
        rf_r = rf_results[rf_results["Dataset"] == ds]
        xgb_r = xgb_results[xgb_results["Dataset"] == ds]
        mlp_r = mlp_results[mlp_results["Dataset"] == ds]
        vae_r = vae_results[vae_results["Dataset"] == ds]
        row = ds
        row += f" & {fmt(lr_r['AUPRC_mean'].values[0], lr_r['AUPRC_std'].values[0]) if not lr_r.empty else '-'}"
        row += f" & {fmt(rf_r['AUPRC_mean'].values[0], rf_r['AUPRC_std'].values[0]) if not rf_r.empty else '-'}"
        row += f" & {fmt(xgb_r['AUPRC_mean'].values[0], xgb_r['AUPRC_std'].values[0]) if not xgb_r.empty else '-'}"
        row += f" & {fmt(mlp_r['AUPRC_mean'].values[0], mlp_r['AUPRC_std'].values[0]) if not mlp_r.empty else '-'}"
        row += f" & {fmt(vae_r['AUPRC_mean'].values[0], vae_r['AUPRC_std'].values[0]) if not vae_r.empty else '-'}"
        row += " \\\\\n"
        latex += row
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"

    latex += "\n\\subsubsection{Социальные сети}\n\n"
    latex += "\\begin{table}[H]\n\\centering\n\\caption{AUPRC на датасетах социальных сетей}\n"
    latex += "\\label{tab:domain_social}\n\\footnotesize\n\\setlength{\\tabcolsep}{4pt}\n"
    latex += "\\begin{tabular}{l" + "c"*len(methods) + "c"*len(graph_methods) + "}\n\\toprule\n"
    latex += "\\textbf{Датасет} & " + " & ".join(methods) + " & " + " & ".join(graph_methods) + " \\\\\n\\midrule\n"
    for ds in social:
        lr_r = lr_results[lr_results["Dataset"] == ds]
        rf_r = rf_results[rf_results["Dataset"] == ds]
        xgb_r = xgb_results[xgb_results["Dataset"] == ds]
        mlp_r = mlp_results[mlp_results["Dataset"] == ds]
        vae_r = vae_results[vae_results["Dataset"] == ds]
        gs_r = graphsage_results[graphsage_results["Dataset"] == ds]
        gcn_r = gcn_results[gcn_results["Dataset"] == ds]
        gat_r = gat_results[gat_results["Dataset"] == ds]
        row = ds
        row += f" & {fmt(lr_r['AUPRC_mean'].values[0], lr_r['AUPRC_std'].values[0]) if not lr_r.empty else '-'}"
        row += f" & {fmt(rf_r['AUPRC_mean'].values[0], rf_r['AUPRC_std'].values[0]) if not rf_r.empty else '-'}"
        row += f" & {fmt(xgb_r['AUPRC_mean'].values[0], xgb_r['AUPRC_std'].values[0]) if not xgb_r.empty else '-'}"
        row += f" & {fmt(mlp_r['AUPRC_mean'].values[0], mlp_r['AUPRC_std'].values[0]) if not mlp_r.empty else '-'}"
        row += f" & {fmt(vae_r['AUPRC_mean'].values[0], vae_r['AUPRC_std'].values[0]) if not vae_r.empty else '-'}"
        row += f" & {fmt(gs_r['AUPRC_mean'].values[0], gs_r['AUPRC_std'].values[0]) if not gs_r.empty else '-'}"
        row += f" & {fmt(gcn_r['AUPRC_mean'].values[0], gcn_r['AUPRC_std'].values[0]) if not gcn_r.empty else '-'}"
        row += f" & {fmt(gat_r['AUPRC_mean'].values[0], gat_r['AUPRC_std'].values[0]) if not gat_r.empty else '-'}"
        row += " \\\\\n"
        latex += row
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    return latex

# ==================== 6.5 ВРЕМЯ ====================
def generate_compute_table():
    methods = ["LR", "RF", "XGBoost", "MLP", "VAE", "GraphSAGE", "GCN", "GAT"]
    latex = "\\begin{table}[H]\n\\centering\n\\caption{Среднее время обучения (секунды)}\n"
    latex += "\\label{tab:compute_time}\n\\footnotesize\n\\setlength{\\tabcolsep}{4pt}\n"
    latex += "\\begin{tabular}{l" + "c"*len(methods) + "}\n\\toprule\n"
    latex += "\\textbf{Датасет} & " + " & ".join(methods) + " \\\\\n\\midrule\n"
    for ds in tabular_datasets:
        lr_r = lr_results[lr_results["Dataset"] == ds]
        rf_r = rf_results[rf_results["Dataset"] == ds]
        xgb_r = xgb_results[xgb_results["Dataset"] == ds]
        mlp_r = mlp_results[mlp_results["Dataset"] == ds]
        vae_r = vae_results[vae_results["Dataset"] == ds]
        row = ds
        row += f" & {fmt_time(lr_r['Train_time_avg'].values[0]) if not lr_r.empty else '-'}"
        row += f" & {fmt_time(rf_r['Train_time_avg'].values[0]) if not rf_r.empty else '-'}"
        row += f" & {fmt_time(xgb_r['Train_time_avg'].values[0]) if not xgb_r.empty else '-'}"
        row += f" & {fmt_time(mlp_r['Train_time_avg'].values[0]) if not mlp_r.empty else '-'}"
        row += f" & {fmt_time(vae_r['Train_time_avg'].values[0]) if not vae_r.empty else '-'}"
        gs_r = graphsage_results[graphsage_results["Dataset"] == ds]
        row += f" & {fmt_time(gs_r['Train_time_avg'].values[0]) if not gs_r.empty else '-'}"
        gcn_r = gcn_results[gcn_results["Dataset"] == ds]
        row += f" & {fmt_time(gcn_r['Train_time_avg'].values[0]) if not gcn_r.empty else '-'}"
        gat_r = gat_results[gat_results["Dataset"] == ds]
        row += f" & {fmt_time(gat_r['Train_time_avg'].values[0]) if not gat_r.empty else '-'}"
        row += " \\\\\n"
        latex += row
    for ds in graph_datasets:
        if ds in tabular_datasets:
            continue
        gs_r = graphsage_results[graphsage_results["Dataset"] == ds]
        gcn_r = gcn_results[gcn_results["Dataset"] == ds]
        gat_r = gat_results[gat_results["Dataset"] == ds]
        row = ds
        row += f" & - & - & - & - & -"
        row += f" & {fmt_time(gs_r['Train_time_avg'].values[0]) if not gs_r.empty else '-'}"
        row += f" & {fmt_time(gcn_r['Train_time_avg'].values[0]) if not gcn_r.empty else '-'}"
        row += f" & {fmt_time(gat_r['Train_time_avg'].values[0]) if not gat_r.empty else '-'}"
        row += " \\\\\n"
        latex += row
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    return latex

if __name__ == "__main__":
    with open("tables_for_diploma.tex", "w", encoding="utf-8") as f:
        f.write("% === ТАБЛИЦЫ ДЛЯ ДИПЛОМА ===\n\n")
        f.write(generate_auprc_table())
        f.write("\n\n")
        f.write(generate_f1_table())
        f.write("\n\n")
        f.write(generate_preck_table())
        f.write("\n\n")
        f.write(generate_domain_tables())
        f.write("\n\n")
        f.write(generate_compute_table())
    print("Таблицы сохранены в tables_for_diploma.tex")

Все файлы успешно загружены
Таблицы сохранены в tables_for_diploma.tex


In [3]:
import pandas as pd
import numpy as np
import glob

# Загружаем все CSV, которые заканчиваются на _results_encoded.csv
files = glob.glob("*_results_encoded.csv")
print("Найдены файлы:", files)

dataframes = {}
for f in files:
    name = f.replace("_results_encoded.csv", "").upper()
    dataframes[name] = pd.read_csv(f)

# Для удобства приведём все к единому формату (столбцы Dataset, AUPRC_mean, AUPRC_std)
for algo, df in dataframes.items():
    # Если есть столбец 'Dataset'
    if 'Dataset' in df.columns:
        pass
    else:
        # возможно, индекс или первый столбец
        df = df.reset_index()
        df.rename(columns={df.columns[0]: 'Dataset'}, inplace=True)
    dataframes[algo] = df

Найдены файлы: ['xgb_results_encoded.csv', 'mlp_results_encoded.csv', 'lr_results_encoded.csv', 'rf_results_encoded.csv', 'vae_results_encoded.csv']


In [14]:
import pandas as pd
import glob

# 1. Загружаем все результаты
files = glob.glob("*_results_encoded.csv")
dataframes = {}
for f in files:
    name = f.replace("_results_encoded.csv", "").upper()
    if name == "LR":
        name = "LR"
    elif name == "RF":
        name = "RF"
    elif name == "XGB":
        name = "XGBoost"
    elif name == "MLP":
        name = "MLP"
    elif name == "VAE":
        name = "VAE"
    dataframes[name] = pd.read_csv(f)

# 2. Желаемый порядок датасетов
dataset_order = [
    "Credit Card Fraud",
    "IEEE-CIS",
    "PaySim",
    "AIZBank",
    "Elliptic (Bitcoin)",
    "Amazon Reviews",
    "Yelp Reviews",
    "Click Fraud",
    "Social Honeypot",
    "MGTAB",
    "Cresci-2017",
    "Midterm-2018"
]

# 3. Желаемый порядок алгоритмов (столбцы)
algo_order = [
    "LR",
    "RF",
    "XGBoost",
    "MLP",
    "VAE"
]

# 4. Собираем AUPRC в нужном порядке
auprc_tables = {}
for algo in algo_order:
    if algo in dataframes:
        df = dataframes[algo]
        if 'AUPRC_mean' in df.columns and 'Dataset' in df.columns:
            df['Dataset'] = df['Dataset'].str.strip()
            auprc_tables[algo] = df[['Dataset', 'AUPRC_mean', 'AUPRC_std']].copy()
            auprc_tables[algo].rename(columns={'AUPRC_mean': f'{algo}_mean', 'AUPRC_std': f'{algo}_std'}, inplace=True)

# 5. Объединяем
from functools import reduce
merged = reduce(lambda left, right: pd.merge(left, right, on='Dataset', how='outer'), auprc_tables.values())
merged = merged.fillna(0)

# 6. Сортируем строки по dataset_order
merged['Dataset'] = pd.Categorical(merged['Dataset'], categories=dataset_order, ordered=True)
merged = merged.sort_values('Dataset').reset_index(drop=True)

# 7. Форматирование
def fmt(row, algo):
    mean = row[f'{algo}_mean']
    std = row[f'{algo}_std']
    if mean == 0 and std == 0:
        return "---"
    else:
        return f"{mean:.4f} $\\pm$ {std:.4f}"

# 8. Генерация LaTeX
latex = "\\begin{table}[H]\n\\centering\n\\caption{Сравнение AUPRC для всех алгоритмов (после кодирования категориальных признаков)}\n"
latex += "\\label{tab:auprc_encoded}\n\\footnotesize\n\\setlength{\\tabcolsep}{3pt}\n"
latex += "\\begin{tabular}{l" + "c"*len(auprc_tables) + "}\n\\toprule\n"
latex += "\\textbf{Датасет} & " + " & ".join(auprc_tables.keys()) + " \\\\\n\\midrule\n"

for idx, row in merged.iterrows():
    ds = row['Dataset']
    line = ds
    for algo in auprc_tables.keys():
        line += f" & {fmt(row, algo)}"
    line += " \\\\\n"
    latex += line
latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"

print(latex)

\begin{table}[H]
\centering
\caption{Сравнение AUPRC для всех алгоритмов (после кодирования категориальных признаков)}
\label{tab:auprc_encoded}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{lccccc}
\toprule
\textbf{Датасет} & LR & RF & XGBoost & MLP & VAE \\
\midrule
Credit Card Fraud & 0.7523 $\pm$ 0.0600 & 0.8406 $\pm$ 0.0304 & 0.8156 $\pm$ 0.0437 & 0.7017 $\pm$ 0.0694 & 0.6258 $\pm$ 0.0902 \\
IEEE-CIS & 0.4049 $\pm$ 0.0037 & 0.6943 $\pm$ 0.0132 & 0.8052 $\pm$ 0.0157 & 0.4106 $\pm$ 0.0052 & 0.1775 $\pm$ 0.1702 \\
PaySim & 0.5606 $\pm$ 0.0120 & 0.9088 $\pm$ 0.0136 & 0.9460 $\pm$ 0.0153 & 0.6338 $\pm$ 0.0276 & 0.2753 $\pm$ 0.0682 \\
AIZBank & 0.5246 $\pm$ 0.0128 & 0.5687 $\pm$ 0.0103 & 0.5733 $\pm$ 0.0170 & 0.5561 $\pm$ 0.0116 & 0.2035 $\pm$ 0.0144 \\
Elliptic (Bitcoin) & 0.9884 $\pm$ 0.0016 & 0.9996 $\pm$ 0.0001 & 0.9996 $\pm$ 0.0001 & 0.9531 $\pm$ 0.0019 & 0.9512 $\pm$ 0.0000 \\
Amazon Reviews & 0.8836 $\pm$ 0.0182 & 0.9086 $\pm$ 0.0231 & 0.9145 $\pm$ 0.0272 & 0.8947 $\p

In [16]:
import pandas as pd
import glob

# 1. Загружаем все результаты
files = glob.glob("*_results_encoded.csv")
dataframes = {}
for f in files:
    name = f.replace("_results_encoded.csv", "").upper()
    if name == "LR":
        name = "LR"
    elif name == "RF":
        name = "RF"
    elif name == "XGB":
        name = "XGBoost"
    elif name == "MLP":
        name = "MLP"
    elif name == "VAE":
        name = "VAE"
    dataframes[name] = pd.read_csv(f)

# 2. Желаемый порядок датасетов
dataset_order = [
    "Credit Card Fraud",
    "IEEE-CIS",
    "PaySim",
    "AIZBank",
    "Elliptic (Bitcoin)",
    "Amazon Reviews",
    "Yelp Reviews",
    "Click Fraud",
    "Social Honeypot",
    "MGTAB",
    "Cresci-2017",
    "Midterm-2018"
]

# 3. Желаемый порядок алгоритмов (столбцы)
algo_order = [
    "LR",
    "RF",
    "XGBoost",
    "MLP",
]

# 4. Собираем F1 для алгоритмов
f1_tables = {}
for algo in algo_order:
    if algo in dataframes:
        df = dataframes[algo]
        if 'F1_mean' in df.columns and 'Dataset' in df.columns:
            df['Dataset'] = df['Dataset'].str.strip()
            f1_tables[algo] = df[['Dataset', 'F1_mean', 'F1_std']].copy()
            f1_tables[algo].rename(columns={'F1_mean': f'{algo}_mean', 'F1_std': f'{algo}_std'}, inplace=True)

# 5. Объединяем
from functools import reduce
merged = reduce(lambda left, right: pd.merge(left, right, on='Dataset', how='outer'), f1_tables.values())
merged = merged.fillna(0)

# 6. Сортируем строки
merged['Dataset'] = pd.Categorical(merged['Dataset'], categories=dataset_order, ordered=True)
merged = merged.sort_values('Dataset').reset_index(drop=True)

# 7. Форматирование
def fmt(row, algo):
    mean = row[f'{algo}_mean']
    std = row[f'{algo}_std']
    if mean == 0 and std == 0:
        return "---"
    else:
        return f"{mean:.4f} $\\pm$ {std:.4f}"

# 8. LaTeX
latex = "\\begin{table}[H]\n\\centering\n\\caption{Сравнение F1-меры для табличных методов (после кодирования категориальных признаков)}\n"
latex += "\\label{tab:f1_encoded}\n\\footnotesize\n\\setlength{\\tabcolsep}{3pt}\n"
latex += "\\begin{tabular}{l" + "c"*len(f1_tables) + "}\n\\toprule\n"
latex += "\\textbf{Датасет} & " + " & ".join(f1_tables.keys()) + " \\\\\n\\midrule\n"

for idx, row in merged.iterrows():
    ds = row['Dataset']
    line = ds
    for algo in f1_tables.keys():
        line += f" & {fmt(row, algo)}"
    line += " \\\\\n"
    latex += line
latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"

print(latex)

\begin{table}[H]
\centering
\caption{Сравнение F1-меры для табличных методов (после кодирования категориальных признаков)}
\label{tab:f1_encoded}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{lcccc}
\toprule
\textbf{Датасет} & LR & RF & XGBoost & MLP \\
\midrule
Credit Card Fraud & 0.7660 $\pm$ 0.0302 & 0.8285 $\pm$ 0.0271 & 0.8231 $\pm$ 0.0418 & 0.7734 $\pm$ 0.0481 \\
IEEE-CIS & 0.4230 $\pm$ 0.0035 & 0.6556 $\pm$ 0.0115 & 0.7584 $\pm$ 0.0154 & 0.4232 $\pm$ 0.0032 \\
PaySim & 0.5646 $\pm$ 0.0089 & 0.8586 $\pm$ 0.0071 & 0.8838 $\pm$ 0.0198 & 0.6160 $\pm$ 0.0190 \\
AIZBank & 0.5491 $\pm$ 0.0120 & 0.5794 $\pm$ 0.0098 & 0.5776 $\pm$ 0.0149 & 0.5644 $\pm$ 0.0147 \\
Elliptic (Bitcoin) & 0.9558 $\pm$ 0.0042 & 0.9942 $\pm$ 0.0010 & 0.9949 $\pm$ 0.0009 & 0.9487 $\pm$ 0.0000 \\
Amazon Reviews & 0.8544 $\pm$ 0.0155 & 0.8557 $\pm$ 0.0259 & 0.8670 $\pm$ 0.0232 & 0.8633 $\pm$ 0.0142 \\
Yelp Reviews & 0.4119 $\pm$ 0.0055 & 0.6943 $\pm$ 0.0119 & 0.7679 $\pm$ 0.0133 & 0.5997 $\pm$ 0.0165 \\

In [18]:
import pandas as pd
import glob

# Порядок датасетов (тот же)
dataset_order = [
    "Credit Card Fraud",
    "IEEE-CIS",
    "PaySim",
    "AIZBank",
    "Elliptic (Bitcoin)",
    "Amazon Reviews",
    "Yelp Reviews",
    "Click Fraud",
    "Social Honeypot",
    "MGTAB",
    "Cresci-2017",
    "Midterm-2018"
]

# Порядок алгоритмов
algo_order = [
    "LR",
    "RF",
    "XGBoost",
    "MLP",
    "VAE"
]

# Загружаем CSV (повторяем загрузку, можно вынести в функцию)
files = glob.glob("*_results_encoded.csv")
dataframes = {}
for f in files:
    name = f.replace("_results_encoded.csv", "").upper()
    if name == "LR":
        name = "LR"
    elif name == "RF":
        name = "RF"
    elif name == "XGB":
        name = "XGBoost"
    elif name == "MLP":
        name = "MLP"
    elif name == "VAE":
        name = "VAE"
    df = pd.read_csv(f)
    if 'Dataset' not in df.columns:
        df = df.rename(columns={df.columns[0]: 'Dataset'})
    df['Dataset'] = df['Dataset'].str.strip()
    dataframes[name] = df

# Собираем Prec@K
pk_tables = {}
for algo in algo_order:
    if algo in dataframes:
        df = dataframes[algo]
        if 'Prec@K_mean' in df.columns and 'Prec@K_std' in df.columns:
            pk_tables[algo] = df[['Dataset', 'Prec@K_mean', 'Prec@K_std']].copy()
            pk_tables[algo].rename(columns={'Prec@K_mean': f'{algo}_mean', 'Prec@K_std': f'{algo}_std'}, inplace=True)
        else:
            pk_tables[algo] = pd.DataFrame(columns=['Dataset', f'{algo}_mean', f'{algo}_std'])
    else:
        pk_tables[algo] = pd.DataFrame(columns=['Dataset', f'{algo}_mean', f'{algo}_std'])

# Объединение
from functools import reduce
merged = reduce(lambda left, right: pd.merge(left, right, on='Dataset', how='outer'), pk_tables.values())
merged = merged.fillna(0)

# Сортировка
merged['Dataset'] = pd.Categorical(merged['Dataset'], categories=dataset_order, ordered=True)
merged = merged.sort_values('Dataset').reset_index(drop=True)

def fmt(row, algo):
    mean = row[f'{algo}_mean']
    std = row[f'{algo}_std']
    if mean == 0 and std == 0:
        return "---"
    else:
        # Для Click Fraud LR std = 0, но это допустимо
        if std == 0:
            return f"{mean:.4f}"
        else:
            return f"{mean:.4f} $\\pm$ {std:.4f}"

# Генерация LaTeX
latex_pk = "\\begin{table}[H]\n\\centering\n\\caption{Сравнение Precision@K для табличных методов (среднее $\\pm$ стандартное отклонение)}\n"
latex_pk += "\\label{tab:preck_encoded}\n\\footnotesize\n\\setlength{\\tabcolsep}{3pt}\n"
latex_pk += "\\begin{tabular}{l" + "c"*len(algo_order) + "}\n\\toprule\n"
latex_pk += "\\textbf{Датасет} & " + " & ".join(algo_order) + " \\\\\n\\midrule\n"

for idx, row in merged.iterrows():
    ds = row['Dataset']
    line = ds
    for algo in algo_order:
        line += f" & {fmt(row, algo)}"
    line += " \\\\\n"
    latex_pk += line
latex_pk += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"

print(latex_pk)

\begin{table}[H]
\centering
\caption{Сравнение Precision@K для табличных методов (среднее $\pm$ стандартное отклонение)}
\label{tab:preck_encoded}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{lccccc}
\toprule
\textbf{Датасет} & LR & RF & XGBoost & MLP & VAE \\
\midrule
Credit Card Fraud & 0.7541 $\pm$ 0.0232 & 0.8108 $\pm$ 0.0320 & 0.7973 $\pm$ 0.0444 & 0.7703 $\pm$ 0.0331 & 0.6757 $\pm$ 0.0598 \\
IEEE-CIS & 0.4185 $\pm$ 0.0037 & 0.6479 $\pm$ 0.0119 & 0.7418 $\pm$ 0.0156 & 0.4110 $\pm$ 0.0027 & 0.1190 $\pm$ 0.0467 \\
PaySim & 0.5291 $\pm$ 0.0114 & 0.8411 $\pm$ 0.0121 & 0.8789 $\pm$ 0.0202 & 0.6028 $\pm$ 0.0290 & 0.2995 $\pm$ 0.0629 \\
AIZBank & 0.5324 $\pm$ 0.0114 & 0.5521 $\pm$ 0.0092 & 0.5609 $\pm$ 0.0140 & 0.5498 $\pm$ 0.0116 & 0.2049 $\pm$ 0.0117 \\
Elliptic (Bitcoin) & 0.9474 $\pm$ 0.0062 & 0.9935 $\pm$ 0.0008 & 0.9945 $\pm$ 0.0006 & 0.8997 $\pm$ 0.0011 & 0.9020 $\pm$ 0.0009 \\
Amazon Reviews & 0.8407 $\pm$ 0.0222 & 0.8537 $\pm$ 0.0178 & 0.8618 $\pm$ 0.0236 & 0.8455 $

In [20]:
import pandas as pd
import glob

# Порядок датасетов
dataset_order = [
    "Credit Card Fraud",
    "IEEE-CIS",
    "PaySim",
    "AIZBank",
    "Elliptic (Bitcoin)",
    "Amazon Reviews",
    "Yelp Reviews",
    "Click Fraud",
    "Social Honeypot",
    "Cresci-2017",
    "Midterm-2018",
    "MGTAB"
]

# Порядок алгоритмов (столбцы)
algo_order = ["LR", "RF", "XGBoost", "MLP", "VAE", "GraphSAGE", "GCN", "GAT"]

# Маппинг имён алгоритмов на соответствующие CSV-файлы и колонку со временем
algo_files = {
    "LR": "lr_results.csv",
    "RF": "rf_results.csv",
    "XGBoost": "xgb_results.csv",
    "MLP": "mlp_results.csv",
    "VAE": "vae_results.csv",
    "GraphSAGE": "graphsage_results.csv",
    "GCN": "gcn_results.csv",
    "GAT": "gat_results.csv"
}

# Загружаем времена для каждого алгоритма
time_data = {algo: {} for algo in algo_order}
for algo, filename in algo_files.items():
    try:
        df = pd.read_csv(filename)
        # Проверяем наличие колонок
        if 'Dataset' not in df.columns:
            # Если первая колонка без имени, переименуем
            df = df.rename(columns={df.columns[0]: 'Dataset'})
        if 'Train_time_avg' in df.columns:
            # Приводим названия датасетов к единому формату (обрезаем пробелы)
            df['Dataset'] = df['Dataset'].str.strip()
            # Заполняем словарь
            for _, row in df.iterrows():
                ds = row['Dataset']
                if ds in dataset_order:  # берём только известные датасеты
                    time_data[algo][ds] = row['Train_time_avg']
    except FileNotFoundError:
        print(f"Файл {filename} не найден, пропускаем.")
    except Exception as e:
        print(f"Ошибка при загрузке {filename}: {e}")

# Формируем LaTeX-таблицу
latex = "\\begin{table}[H]\n\\centering\n\\caption{Среднее время обучения (секунды)}\n"
latex += "\\label{tab:compute_time}\n\\footnotesize\n\\setlength{\\tabcolsep}{4pt}\n"
latex += "\\begin{tabular}{l" + "c"*len(algo_order) + "}\n\\toprule\n"
latex += "\\textbf{Датасет} & " + " & ".join(algo_order) + " \\\\\n\\midrule\n"

for ds in dataset_order:
    row = ds
    for algo in algo_order:
        time_val = time_data[algo].get(ds)
        if time_val is not None:
            # Округляем до двух знаков, если целое – убираем .00
            if isinstance(time_val, (int, float)):
                if time_val == int(time_val):
                    row += f" & {int(time_val)}"
                else:
                    row += f" & {time_val:.2f}"
            else:
                row += f" & {time_val}"
        else:
            row += " & --"
    row += " \\\\\n"
    latex += row

latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"

print(latex)

\begin{table}[H]
\centering
\caption{Среднее время обучения (секунды)}
\label{tab:compute_time}
\footnotesize
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lcccccccc}
\toprule
\textbf{Датасет} & LR & RF & XGBoost & MLP & VAE & GraphSAGE & GCN & GAT \\
\midrule
Credit Card Fraud & 0.23 & 50.94 & 1.02 & 3.36 & 10.61 & -- & -- & -- \\
IEEE-CIS & 39.29 & 79.34 & 16.50 & 9.72 & 2.01 & -- & -- & -- \\
PaySim & 7.60 & 512.33 & 21.50 & 19.77 & 9.59 & -- & -- & -- \\
AIZBank & 0.05 & 2.14 & 0.60 & 8.22 & 8.46 & -- & -- & -- \\
Elliptic (Bitcoin) & 2.53 & 6.70 & 2.01 & 2.11 & 0.18 & 19.96 & 16.39 & 31.08 \\
Amazon Reviews & 0.01 & 0.81 & 37.77 & 1.88 & 3.53 & 80.74 & 240.99 & 214.17 \\
Yelp Reviews & 0.44 & 4.41 & 2.62 & 8.10 & 9.36 & 18.22 & 24.31 & 68.14 \\
Click Fraud & 0.10 & 3.80 & 0.47 & 5.67 & 1.35 & -- & -- & -- \\
Social Honeypot & 0.02 & 2.47 & 0.63 & 6.98 & 1.54 & -- & -- & -- \\
Cresci-2017 & 0.09 & 0.66 & 0.22 & 0.91 & 0.28 & -- & -- & -- \\
Midterm-2018 & 0.04 & 2.52 & 0.36 & 2.53 & 

In [28]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Порядок датасетов
# ------------------------------------------------------------
dataset_order = [
    "Credit Card Fraud",
    "IEEE-CIS",
    "PaySim",
    "AIZBank",
    "Elliptic (Bitcoin)",
    "Amazon Reviews",
    "Yelp Reviews",
    "Click Fraud",
    "Social Honeypot",
    "Cresci-2017",
    "Midterm-2018",
    "MGTAB"
]

# ------------------------------------------------------------
# Загрузка CSV
# ------------------------------------------------------------
def load_csv(fname):
    try:
        df = pd.read_csv(fname)
        if 'Dataset' not in df.columns:
            df = df.rename(columns={df.columns[0]: 'Dataset'})
        df['Dataset'] = df['Dataset'].str.strip()
        return df
    except:
        return None

lr_df = load_csv("lr_results_encoded.csv")
rf_df = load_csv("rf_results_encoded.csv")
xgb_df = load_csv("xgb_results_encoded.csv")
mlp_df = load_csv("mlp_results_encoded.csv")
vae_df = load_csv("vae_results_encoded.csv")
gs_df = load_csv("graphsage_results.csv")
gcn_df = load_csv("gcn_results.csv")
gat_df = load_csv("gat_results.csv")

# ------------------------------------------------------------
# Функция форматирования одного числа (без ±)
# ------------------------------------------------------------
def fmt_val(x, digits=4):
    if pd.isna(x):
        return "--"
    else:
        return f"{x:.{digits}f}"

# ------------------------------------------------------------
# 1. Логистическая регрессия (LR)
# ------------------------------------------------------------
def lr_table():
    if lr_df is None:
        return ""
    latex = "\\begin{table}[H]\n\\centering\n"
    latex += "\\caption{Метрики качества логистической регрессии на всех датасетах (среднее $\\pm$ стандартное отклонение по 5 запускам)}\n"
    latex += "\\label{tab:lr_results_compact}\n\\scriptsize\n\\setlength{\\tabcolsep}{4pt}\n"
    latex += "\\begin{tabular}{lccccccccc}\n\\toprule\n"
    latex += "\\textbf{Датасет} & \\multicolumn{2}{c}{\\textbf{AUPRC}} & \\multicolumn{2}{c}{\\textbf{AUROC}} & \\multicolumn{2}{c}{\\textbf{F1}} & \\multicolumn{2}{c}{\\textbf{Prec@K}} & \\textbf{Best $C$} \\\\\n"
    latex += "\\cmidrule(lr){2-3} \\cmidrule(lr){4-5} \\cmidrule(lr){6-7} \\cmidrule(lr){8-9}\n"
    latex += "& mean & std & mean & std & mean & std & mean & std & \\\\\n\\midrule\n"
    for ds in dataset_order:
        row = lr_df[lr_df['Dataset'] == ds]
        if row.empty:
            continue
        r = row.iloc[0]
        best_c = r.get('Best_C', '---')
        if pd.isna(best_c) or best_c == '':
            best_c = '---'
        else:
            best_c = f"{best_c:.2f}" if isinstance(best_c, (int, float)) else str(best_c)
        latex += f"{ds} & {fmt_val(r['AUPRC_mean'])} & {fmt_val(r['AUPRC_std'])} & {fmt_val(r['AUROC_mean'])} & {fmt_val(r['AUROC_std'])} & {fmt_val(r['F1_mean'])} & {fmt_val(r['F1_std'])} & {fmt_val(r['Prec@K_mean'])} & {fmt_val(r['Prec@K_std'])} & {best_c} \\\\\n"
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    return latex

# ------------------------------------------------------------
# 2. Случайный лес (RF)
# ------------------------------------------------------------
def rf_table():
    if rf_df is None:
        return ""
    def param_str(r):
        d = r.get('Best_max_depth', '')
        l = r.get('Best_min_samples_leaf', '')
        f = r.get('Best_max_features', '')
        parts = []
        if not pd.isna(d) and d != '' and d != 'nan':
            parts.append(f"d={d}")
        if not pd.isna(l) and l != '' and l != 'nan':
            parts.append(f"l={l}")
        if not pd.isna(f) and f != '' and f != 'nan':
            parts.append(f"{f}")
        return ", ".join(parts) if parts else "---"
    latex = "\\begin{table}[H]\n\\centering\n"
    latex += "\\caption{Метрики качества случайного леса на всех датасетах (среднее $\\pm$ стандартное отклонение по 5 запускам)}\n"
    latex += "\\label{tab:rf_results_compact}\n\\scriptsize\n\\setlength{\\tabcolsep}{4pt}\n"
    latex += "\\begin{tabular}{lccccccccc}\n\\toprule\n"
    latex += "\\textbf{Датасет} & \\multicolumn{2}{c}{\\textbf{AUPRC}} & \\multicolumn{2}{c}{\\textbf{AUROC}} & \\multicolumn{2}{c}{\\textbf{F1}} & \\multicolumn{2}{c}{\\textbf{Prec@K}} & \\textbf{Лучшие параметры} \\\\\n"
    latex += "\\cmidrule(lr){2-3} \\cmidrule(lr){4-5} \\cmidrule(lr){6-7} \\cmidrule(lr){8-9}\n"
    latex += "& mean & std & mean & std & mean & std & mean & std & \\\\\n\\midrule\n"
    for ds in dataset_order:
        row = rf_df[rf_df['Dataset'] == ds]
        if row.empty:
            continue
        r = row.iloc[0]
        latex += f"{ds} & {fmt_val(r['AUPRC_mean'])} & {fmt_val(r['AUPRC_std'])} & {fmt_val(r['AUROC_mean'])} & {fmt_val(r['AUROC_std'])} & {fmt_val(r['F1_mean'])} & {fmt_val(r['F1_std'])} & {fmt_val(r['Prec@K_mean'])} & {fmt_val(r['Prec@K_std'])} & {param_str(r)} \\\\\n"
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    return latex

# ------------------------------------------------------------
# 3. XGBoost (метрики, без времени)
# ------------------------------------------------------------
def xgb_metrics_table():
    if xgb_df is None:
        return ""
    latex = "\\begin{table}[H]\n\\centering\n"
    latex += "\\caption{Метрики качества XGBoost (среднее $\\pm$ стандартное отклонение по 5 запускам)}\n"
    latex += "\\label{tab:xgb_metrics}\n\\scriptsize\n\\setlength{\\tabcolsep}{7pt}\n"
    latex += "\\begin{tabular}{lcccccccc}\n\\toprule\n"
    latex += "\\textbf{Датасет} & \\multicolumn{2}{c}{\\textbf{AUPRC}} & \\multicolumn{2}{c}{\\textbf{AUROC}} & \\multicolumn{2}{c}{\\textbf{F1}} & \\multicolumn{2}{c}{\\textbf{Prec@K}} \\\\\n"
    latex += "\\cmidrule(lr){2-3} \\cmidrule(lr){4-5} \\cmidrule(lr){6-7} \\cmidrule(lr){8-9}\n"
    latex += "& mean & std & mean & std & mean & std & mean & std \\\\\n\\midrule\n"
    for ds in dataset_order:
        row = xgb_df[xgb_df['Dataset'] == ds]
        if row.empty:
            continue
        r = row.iloc[0]
        latex += f"{ds} & {fmt_val(r['AUPRC_mean'])} & {fmt_val(r['AUPRC_std'])} & {fmt_val(r['AUROC_mean'])} & {fmt_val(r['AUROC_std'])} & {fmt_val(r['F1_mean'])} & {fmt_val(r['F1_std'])} & {fmt_val(r['Prec@K_mean'])} & {fmt_val(r['Prec@K_std'])} \\\\\n"
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    return latex

# ------------------------------------------------------------
# 4. XGBoost гиперпараметры (отдельная таблица)
# ------------------------------------------------------------
def xgb_params_table():
    if xgb_df is None:
        return ""
    needed = ['learning_rate', 'n_estimators', 'max_depth', 'reg_lambda', 'subsample', 'colsample_bytree']
    if not all(c in xgb_df.columns for c in needed):
        return "% Гиперпараметры XGBoost отсутствуют\n"
    latex = "\\begin{table}[H]\n\\centering\n"
    latex += "\\caption{Лучшие гиперпараметры XGBoost (найдены по максимуму AUPRC на валидации)}\n"
    latex += "\\label{tab:xgb_params}\n\\scriptsize\n\\setlength{\\tabcolsep}{3.5pt}\n"
    latex += "\\begin{tabular}{lcccccc}\n\\toprule\n"
    latex += "\\textbf{Датасет} & \\textbf{learning\\_rate} & \\textbf{n\\_estimators} & \\textbf{max\\_depth} & \\textbf{reg\\_lambda} & \\textbf{subsample} & \\textbf{colsample\\_bytree} \\\\\n\\midrule\n"
    for ds in dataset_order:
        row = xgb_df[xgb_df['Dataset'] == ds]
        if row.empty:
            continue
        r = row.iloc[0]
        latex += f"{ds} & {r['learning_rate']} & {r['n_estimators']} & {r['max_depth']} & {r['reg_lambda']} & {r['subsample']} & {r['colsample_bytree']} \\\\\n"
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    return latex

# ------------------------------------------------------------
# 5. MLP (метрики)
# ------------------------------------------------------------
def mlp_table():
    if mlp_df is None:
        return ""
    latex = "\\begin{table}[H]\n\\centering\n"
    latex += "\\caption{Метрики качества MLP на всех датасетах (среднее $\\pm$ стандартное отклонение по 5 запускам)}\n"
    latex += "\\label{tab:mlp_results}\n\\scriptsize\n\\setlength{\\tabcolsep}{7pt}\n"
    latex += "\\begin{tabular}{lcccccccc}\n\\toprule\n"
    latex += "\\textbf{Датасет} & \\multicolumn{2}{c}{\\textbf{AUPRC}} & \\multicolumn{2}{c}{\\textbf{AUROC}} & \\multicolumn{2}{c}{\\textbf{F1}} & \\multicolumn{2}{c}{\\textbf{Prec@K}} \\\\\n"
    latex += "\\cmidrule(lr){2-3} \\cmidrule(lr){4-5} \\cmidrule(lr){6-7} \\cmidrule(lr){8-9}\n"
    latex += "& mean & std & mean & std & mean & std & mean & std \\\\\n\\midrule\n"
    for ds in dataset_order:
        row = mlp_df[mlp_df['Dataset'] == ds]
        if row.empty:
            continue
        r = row.iloc[0]
        latex += f"{ds} & {fmt_val(r['AUPRC_mean'])} & {fmt_val(r['AUPRC_std'])} & {fmt_val(r['AUROC_mean'])} & {fmt_val(r['AUROC_std'])} & {fmt_val(r['F1_mean'])} & {fmt_val(r['F1_std'])} & {fmt_val(r['Prec@K_mean'])} & {fmt_val(r['Prec@K_std'])} \\\\\n"
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    return latex

# ------------------------------------------------------------
# 6. VAE (без F1)
# ------------------------------------------------------------
def vae_table():
    if vae_df is None:
        return ""
    latex = "\\begin{table}[H]\n\\centering\n"
    latex += "\\caption{Метрики качества VAE на всех датасетах (среднее $\\pm$ стандартное отклонение по 5 запускам)}\n"
    latex += "\\label{tab:vae_results}\n\\footnotesize\n\\setlength{\\tabcolsep}{4pt}\n"
    latex += "\\begin{tabular}{lcccccc}\n\\toprule\n"
    latex += "\\textbf{Датасет} & \\multicolumn{2}{c}{\\textbf{AUPRC}} & \\multicolumn{2}{c}{\\textbf{AUROC}} & \\multicolumn{2}{c}{\\textbf{Prec@K}} \\\\\n"
    latex += "\\cmidrule(lr){2-3} \\cmidrule(lr){4-5} \\cmidrule(lr){6-7}\n"
    latex += "& mean & std & mean & std & mean & std \\\\\n\\midrule\n"
    for ds in dataset_order:
        row = vae_df[vae_df['Dataset'] == ds]
        if row.empty:
            continue
        r = row.iloc[0]
        latex += f"{ds} & {fmt_val(r['AUPRC_mean'])} & {fmt_val(r['AUPRC_std'])} & {fmt_val(r['AUROC_mean'])} & {fmt_val(r['AUROC_std'])} & {fmt_val(r['Prec@K_mean'])} & {fmt_val(r['Prec@K_std'])} \\\\\n"
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    return latex

# ------------------------------------------------------------
# 7. GCN, GAT, GraphSAGE
# ------------------------------------------------------------
def graph_table(name, df, label):
    if df is None:
        return ""
    latex = f"\\begin{{table}}[H]\n\\centering\n"
    latex += f"\\caption{{Метрики качества {label} на графовых датасетах (среднее $\\pm$ стандартное отклонение по 5 запускам)}}\n"
    latex += f"\\label{{tab:{name.lower()}_results}}\n\\footnotesize\n\\setlength{{\\tabcolsep}}{{4pt}}\n"
    latex += "\\begin{tabular}{lcccccc}\n\\toprule\n"
    latex += "\\textbf{Датасет} & \\multicolumn{2}{c}{\\textbf{AUPRC}} & \\multicolumn{2}{c}{\\textbf{AUROC}} & \\multicolumn{2}{c}{\\textbf{Prec@K}} \\\\\n"
    latex += "\\cmidrule(lr){2-3} \\cmidrule(lr){4-5} \\cmidrule(lr){6-7}\n"
    latex += "& mean & std & mean & std & mean & std \\\\\n\\midrule\n"
    for ds in ["Elliptic (Bitcoin)", "Amazon Reviews", "Yelp Reviews", "MGTAB"]:
        row = df[df['Dataset'] == ds]
        if row.empty:
            continue
        r = row.iloc[0]
        latex += f"{ds} & {fmt_val(r['AUPRC_mean'])} & {fmt_val(r['AUPRC_std'])} & {fmt_val(r['AUROC_mean'])} & {fmt_val(r['AUROC_std'])} & {fmt_val(r['Prec@K_mean'])} & {fmt_val(r['Prec@K_std'])} \\\\\n"
    latex += "\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    return latex

# ------------------------------------------------------------
# Вывод
# ------------------------------------------------------------
print(lr_table())
print(rf_table())
print(xgb_metrics_table())
print(xgb_params_table())
print(mlp_table())
print(vae_table())
print(graph_table("GCN", gcn_df, "GCN"))
print(graph_table("GAT", gat_df, "GAT"))
print(graph_table("GraphSAGE", gs_df, "GraphSAGE"))

\begin{table}[H]
\centering
\caption{Метрики качества логистической регрессии на всех датасетах (среднее $\pm$ стандартное отклонение по 5 запускам)}
\label{tab:lr_results_compact}
\scriptsize
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lccccccccc}
\toprule
\textbf{Датасет} & \multicolumn{2}{c}{\textbf{AUPRC}} & \multicolumn{2}{c}{\textbf{AUROC}} & \multicolumn{2}{c}{\textbf{F1}} & \multicolumn{2}{c}{\textbf{Prec@K}} & \textbf{Best $C$} \\
\cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9}
& mean & std & mean & std & mean & std & mean & std & \\
\midrule
Credit Card Fraud & 0.7523 & 0.0600 & 0.9793 & 0.0075 & 0.7660 & 0.0302 & 0.7541 & 0.0232 & 1.00 \\
IEEE-CIS & 0.4049 & 0.0037 & 0.8634 & 0.0015 & 0.4230 & 0.0035 & 0.4185 & 0.0037 & 10.00 \\
PaySim & 0.5606 & 0.0120 & 0.9897 & 0.0013 & 0.5646 & 0.0089 & 0.5291 & 0.0114 & 0.01 \\
AIZBank & 0.5246 & 0.0128 & 0.8922 & 0.0052 & 0.5491 & 0.0120 & 0.5324 & 0.0114 & 100.00 \\
Elliptic (Bitcoin) & 0.9884 & 0.0016 & 0.

In [33]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import confusion_matrix, precision_recall_curve
from tqdm import tqdm
import ast

SEED = [42, 43, 44, 45, 46]
TRAIN_SIZE = 0.7
VAL_SIZE = 0.15
TEST_SIZE = 0.15
SUBSAMPLE_LIMIT = 60000   # для RF и XGB
DATA_ROOT = "final_data_encoded"

dataset_order = [
    "Credit Card Fraud",
    "IEEE-CIS",
    "PaySim",
    "AIZBank",
    "Elliptic (Bitcoin)",
    "Amazon Reviews",
    "Yelp Reviews",
    "Click Fraud",
    "Social Honeypot",
    "Cresci-2017",
    "Midterm-2018",
    "MGTAB"
]

def load_best_params():
    lr_df = pd.read_csv("lr_results_encoded.csv")
    lr_params = {row['Dataset']: row.get('Best_C', 1.0) for _, row in lr_df.iterrows()}

    rf_df = pd.read_csv("rf_results_encoded.csv")
    rf_params = {}
    for _, row in rf_df.iterrows():
        ds = row['Dataset']
        max_depth = row.get('Best_max_depth', None)
        if pd.isna(max_depth) or max_depth == 'nan':
            max_depth = None
        else:
            max_depth = int(max_depth)
        min_samples_leaf = int(row.get('Best_min_samples_leaf', 1))
        max_features = row.get('Best_max_features', 'sqrt')
        if max_features == 'nan':
            max_features = None
        rf_params[ds] = {'max_depth': max_depth, 'min_samples_leaf': min_samples_leaf, 'max_features': max_features}

    xgb_df = pd.read_csv("xgb_results_encoded.csv")
    xgb_params = {}
    for _, row in xgb_df.iterrows():
        ds = row['Dataset']
        best_params_str = row['Best_params']
        best_params_str = best_params_str.replace("np.float64(", "").replace(")", "")
        best_params_str = best_params_str.replace("np.int64(", "").replace(")", "")
        best_params = ast.literal_eval(best_params_str)
        xgb_params[ds] = {
            'learning_rate': best_params.get('learning_rate', 0.1),
            'n_estimators': best_params.get('n_estimators', 100),
            'max_depth': best_params.get('max_depth', 3),
            'reg_lambda': best_params.get('reg_lambda', 1.0),
            'subsample': best_params.get('subsample', 1.0),
            'colsample_bytree': best_params.get('colsample_bytree', 1.0),
        }
    return lr_params, rf_params, xgb_params

lr_params, rf_params, xgb_params = load_best_params()

def load_encoded_dataset(name):
    X = np.load(os.path.join(DATA_ROOT, f"{name.replace(' ', '_')}_X.npy"))
    y = np.load(os.path.join(DATA_ROOT, f"{name.replace(' ', '_')}_y.npy"))
    return X, y

def split_stratified(X, y, train_ratio, val_ratio, test_ratio, random_state):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=val_ratio+test_ratio, stratify=y, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=test_ratio/(val_ratio+test_ratio),
        stratify=y_temp, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

def evaluate_model(model, X_val, y_val, X_test, y_test):
    proba_val = model.predict_proba(X_val)[:, 1]
    prec, rec, thresholds = precision_recall_curve(y_val, proba_val)
    f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_thr = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    proba_test = model.predict_proba(X_test)[:, 1]
    y_pred = (proba_test >= best_thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    return tp, tn, fp, fn

results = []
for ds in tqdm(dataset_order):
    X, y = load_encoded_dataset(ds)
    print(f"\n{ds}")
    X_train, X_val, X_test, y_train, y_val, y_test = split_stratified(
        X, y, TRAIN_SIZE, VAL_SIZE, TEST_SIZE, SEED
    )
    if ds in lr_params:
        lr = LogisticRegression(C=lr_params[ds], class_weight='balanced', max_iter=1000, solver='lbfgs', random_state=SEED)
        lr.fit(X_train, y_train)
        tp, tn, fp, fn = evaluate_model(lr, X_val, y_val, X_test, y_test)
        results.append({'Dataset': ds, 'Algorithm': 'LR', 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn})
    if ds in rf_params:
        rf = RandomForestClassifier(n_estimators=500, class_weight='balanced_subsample', n_jobs=-1, random_state=SEED, **rf_params[ds])
        if len(X_train) > SUBSAMPLE_LIMIT:
            X_train_sub, _, y_train_sub, _ = train_test_split(X_train, y_train, train_size=SUBSAMPLE_LIMIT, stratify=y_train, random_state=SEED)
        else:
            X_train_sub, y_train_sub = X_train, y_train
        rf.fit(X_train_sub, y_train_sub)
        tp, tn, fp, fn = evaluate_model(rf, X_val, y_val, X_test, y_test)
        results.append({'Dataset': ds, 'Algorithm': 'RF', 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn})
    # Обучение XGB с подвыборкой
    if ds in xgb_params:
        p = xgb_params[ds]
        scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
        xgb_model = xgb.XGBClassifier(
            learning_rate=p['learning_rate'], n_estimators=p['n_estimators'], max_depth=p['max_depth'],
            reg_lambda=p['reg_lambda'], subsample=p['subsample'], colsample_bytree=p['colsample_bytree'],
            objective='binary:logistic', tree_method='hist', scale_pos_weight=scale_pos_weight,
            random_state=SEED, verbosity=0
        )
        if len(X_train) > SUBSAMPLE_LIMIT:
            X_train_sub, _, y_train_sub, _ = train_test_split(X_train, y_train, train_size=SUBSAMPLE_LIMIT, stratify=y_train, random_state=SEED)
        else:
            X_train_sub, y_train_sub = X_train, y_train
        xgb_model.fit(X_train_sub, y_train_sub)
        tp, tn, fp, fn = evaluate_model(xgb_model, X_val, y_val, X_test, y_test)
        results.append({'Dataset': ds, 'Algorithm': 'XGB', 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn})

df_cm = pd.DataFrame(results)
df_cm.to_csv("confusion_matrices_fast.csv", index=False)
print("Сохранено confusion_matrices_fast.csv")

df_cm['FPR'] = df_cm['FP'] / (df_cm['FP'] + df_cm['TN'])
df_cm['FNR'] = df_cm['FN'] / (df_cm['FN'] + df_cm['TP'])
print(df_cm[['Dataset', 'Algorithm', 'FPR', 'FNR']].round(4))ам

  0%|          | 0/12 [00:00<?, ?it/s]


Credit Card Fraud


  8%|▊         | 1/12 [00:04<00:45,  4.12s/it]


IEEE-CIS


 17%|█▋        | 2/12 [00:57<05:31, 33.19s/it]

...


Midterm-2018


 92%|█████████▏| 11/12 [01:40<00:03,  3.45s/it]


MGTAB


100%|██████████| 12/12 [02:01<00:00, 10.11s/it]

Сохранено confusion_matrices_fast.csv
               Dataset Algorithm     FPR     FNR
0    Credit Card Fraud        LR  0.0002  0.2703
1    Credit Card Fraud        RF  0.0002  0.2162
2    Credit Card Fraud       XGB  0.0002  0.2162
3             IEEE-CIS        LR  0.0169  0.6131
4             IEEE-CIS        RF  0.0112  0.5576
5             IEEE-CIS       XGB  0.0072  0.5363
6               PaySim        LR  0.0002  0.5706
7               PaySim        RF  0.0001  0.3255
8               PaySim       XGB  0.0001  0.3328
9              AIZBank        LR  0.0918  0.3770
10             AIZBank        RF  0.0865  0.3405
11             AIZBank       XGB  0.0710  0.4149
12  Elliptic (Bitcoin)        LR  0.5029  0.0187
13  Elliptic (Bitcoin)        RF  0.0660  0.0019
14  Elliptic (Bitcoin)       XGB  0.0616  0.0014
15      Amazon Reviews        LR  0.0054  0.1789
16      Amazon Reviews        RF  0.0036  0.2195
17      Amazon Reviews       XGB  0.0030  0.1951
18        Yelp Reviews        L

In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# ------------------------------------------------------------
# 1. Загружаем мета-признаки
# ------------------------------------------------------------
meta = pd.DataFrame({
    'Dataset': [
        "Credit Card Fraud", "IEEE-CIS", "PaySim", "AIZBank",
        "Elliptic (Bitcoin)", "Amazon Reviews", "Yelp Reviews",
        "Click Fraud", "Social Honeypot", "MGTAB", "Cresci-2017", "Midterm-2018"
    ],
    'log10_N': [5.45, 5.77, 6.80, 4.66, 4.67, 4.08, 4.66, 5.00, 4.62, 4.01, 3.79, 4.70],
    'log10_IR': [2.76, 1.44, 2.89, 0.88, -0.97, 1.13, 0.77, 2.64, -0.06, 0.43, 0.12, -0.72],
    'd': [30, 393, 10, 17, 168, 25, 32, 7, 6, 788, 21, 19],
    'G': [0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0]
})

# ------------------------------------------------------------
# 2. Загружаем результаты AUPRC (как у вас)
# ------------------------------------------------------------
def load_auprc(fname):
    try:
        df = pd.read_csv(fname)
        if 'AUPRC_mean' in df.columns:
            return df.set_index('Dataset')['AUPRC_mean'].to_dict()
        elif 'AUPRC' in df.columns:
            return df.set_index('Dataset')['AUPRC'].to_dict()
        else:
            return {}
    except:
        return {}

lr_auprc = load_auprc("lr_results_encoded.csv")
rf_auprc = load_auprc("rf_results_encoded.csv")
xgb_auprc = load_auprc("xgb_results_encoded.csv")
mlp_auprc = load_auprc("mlp_results_encoded.csv")
vae_auprc = load_auprc("vae_results_encoded.csv")
gs_auprc = load_auprc("graphsage_results.csv")
gcn_auprc = load_auprc("gcn_results.csv")
gat_auprc = load_auprc("gat_results.csv")

all_metrics = pd.DataFrame(meta)
for name, d in [('LR', lr_auprc), ('RF', rf_auprc), ('XGB', xgb_auprc),
                ('MLP', mlp_auprc), ('VAE', vae_auprc),
                ('GraphSAGE', gs_auprc), ('GCN', gcn_auprc), ('GAT', gat_auprc)]:
    all_metrics[name] = all_metrics['Dataset'].map(d).fillna(np.nan)

all_metrics['XGB_minus_LR'] = all_metrics['XGB'] - all_metrics['LR']
all_metrics['MLP_minus_RF'] = all_metrics['MLP'] - all_metrics['RF']

# ------------------------------------------------------------
# 3. Корреляционный анализ с поправкой Бонферрони
# ------------------------------------------------------------
meta_cols = ['log10_N', 'log10_IR', 'd', 'G']
target_cols = ['XGB', 'XGB_minus_LR', 'MLP', 'MLP_minus_RF', 'GraphSAGE']

corr_results = []
m_total = 0

# Сначала считаем все корреляции, чтобы узнать общее число проверок
for m in meta_cols:
    for t in target_cols:
        mask = all_metrics[[m, t]].notna().all(axis=1)
        if mask.sum() >= 3:
            m_total += 1

alpha = 0.05
alpha_bonferroni = alpha / m_total

print(f"Всего проверок: {m_total}")
print(f"Порог значимости после поправки Бонферрони: α = {alpha_bonferroni:.5f}\n")

for m in meta_cols:
    for t in target_cols:
        mask = all_metrics[[m, t]].notna().all(axis=1)
        if mask.sum() < 3:
            continue
        rho, p = stats.spearmanr(all_metrics.loc[mask, m], all_metrics.loc[mask, t])
        corr_results.append({
            'Мета-признак': m,
            'Метрика': t,
            'ρ': round(rho, 3),
            'p-значение': round(p, 5),
            'значимо (p<0.05)': p < alpha,
            'значимо (Бонферрони)': p < alpha_bonferroni
        })

df_corr = pd.DataFrame(corr_results)
print("Корреляции Спирмена с поправкой Бонферрони:\n")
print(df_corr.to_string(index=False))

# ------------------------------------------------------------
# 4. Вывод только значимых после поправки
# ------------------------------------------------------------
bonf_sign = df_corr[df_corr['значимо (Бонферрони)'] == True]
print("\nКорреляции, значимые после поправки Бонферрони:\n")
if bonf_sign.empty:
    print("Нет ни одной значимой корреляции после поправки.")
else:
    print(bonf_sign.to_string(index=False))

Всего проверок: 20
Порог значимости после поправки Бонферрони: α = 0.00250

Корреляции Спирмена с поправкой Бонферрони:

Мета-признак      Метрика      ρ  p-значение  значимо (p<0.05)  значимо (Бонферрони)
     log10_N          XGB -0.270     0.39658             False                 False
     log10_N XGB_minus_LR  0.441     0.15093             False                 False
     log10_N          MLP -0.469     0.12371             False                 False
     log10_N MLP_minus_RF -0.602     0.03816              True                 False
     log10_N    GraphSAGE  0.200     0.80000             False                 False
    log10_IR          XGB -0.685     0.01391              True                 False
    log10_IR XGB_minus_LR  0.685     0.01391              True                 False
    log10_IR          MLP -0.783     0.00259              True                 False
    log10_IR MLP_minus_RF -0.720     0.00824              True                 False
    log10_IR    GraphSAGE -0.

/var/folders/md/78mwsg9n23525_j8040kg6fw0000gn/T/ipykernel_5551/4228181556.py:80: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = stats.spearmanr(all_metrics.loc[mask, m], all_metrics.loc[mask, t])
